# Day 12 — NumPy Mini-Project: Sensor Telemetry Data Processor

This notebook covers End-to-End Vectorized Sensor Telemetry Data Pipeline

## Validation goals
- Generate synthetic time-series telemetry data across multi-sensor channels without Python loops
- Clean missing data signals and outliers using compound boolean masks (`&`, `|`)
- Execute in-place replacements using median value imputation on valid subsets
- Perform spatial and temporal aggregations across specific axes (`axis=0`)
- Normalize time-series features using vectorized $Z$-score standardization and broadcasting

### Sensor Telemetry Data Processor

 We'll work through designing and implementing this mini-project step-by-step using pure NumPy. 
 Here is the plan for our data pipeline:
 - Data Generation & Structuring: Create a simulated 2D dataset of sensor readings over time ($N \text{ time steps} \times M \text{ sensors}$).
 - Data Cleaning & Anomaly Detection: Identify missing values or extreme noise using vectorized boolean masks and replace them cleanly.
 - Statistical Aggregation & Normalization: Compute spatial trends (per sensor across time) and temporal trends (across sensors per time step) using axis-based aggregations, followed by $Z$-score normalization.

### Step 1: Data Generation
Let's generate a synthetic matrix representing 5 sensors recording data across 10 time steps:

### Guiding Question:

To clean raw_data, we need to replace all anomalous values (readings $\le 0^\circ\text{C}$ or $\ge 100^\circ\text{C}$) with the median value of the valid readings.How would you write the boolean masking expression to calculate the median of only the valid readings in raw_data?

In [3]:
import numpy as np

# Seed for reproducibility
np.random.seed(42)

# Raw temperature readings (5 sensors, 10 time steps)
raw_data = np.random.normal(loc=25.0, scale=3.0, size=(10, 5))

# Injecting artificial anomalies (outliers / missing data signals)
raw_data[2, 1] = -999.0  # Sensor failure code
raw_data[7, 3] = 150.0   # Thermal spike anomaly

# Use boolean masking to select values that are greater than 0 and less than 100, then calculate their median with np.median()
valid_median = np.median(raw_data[(raw_data > 0) & (raw_data < 100)])

How it works:
- raw_data > 0 selects readings above \(0^\circ C\).
- raw_data < 100 selects readings below \(100^\circ C\).
- & combines both conditions element-wise.
- raw_data[...] extracts only the valid readings.
- np.median(...) calculates their median.
Why use & instead of and? NumPy arrays require element-wise logical operators such as &. Each condition must be enclosed in parentheses.
This excludes both -999.0 and 150.0 from the median calculation.

Apply that median back to `raw_data` to clean up the anomalies in-place.

In [ ]:
# Create a copy for cleaning
clean_data = raw_data.copy()

# Define the anomaly mask
anomaly_mask = (clean_data <= 0) | (clean_data >= 100)

### Guiding Question:

Using `anomaly_mask` and the `valid_median` value you just calculated, how would you perform the in-place replacement to overwrite only those bad values in clean_data?

In [ ]:
clean_data[anomaly_mask] = valid_median

How it works:
- anomaly_mask is True wherever a reading is `<= 0 or >= 100`.
- clean_data[anomaly_mask] selects only those anomalous elements.
- = `valid_median` overwrites those elements with the median of valid readings.
Because you're assigning directly to clean_data, the replacement happens in place. All valid readings remain unchanged.

Combining bitwise operators with explicit parentheses filters out both `-999.0` and `150.0` perfectly to compute the clean `median (25.04)`.

Apply that median back to `raw_data` to clean up the anomalies in-place.

In [ ]:
# Create a copy for cleaning
clean_data = raw_data.copy()

# Define the anomaly mask
anomaly_mask = (clean_data <= 0) | (clean_data >= 100)

### Guiding Question:

Using `anomaly_mask` and the `valid_median` value you just calculated, how would you perform the in-place replacement to overwrite only those bad values in clean_data?

In [ ]:
clean_data[anomaly_mask] = valid_median

By passing `anomaly_mask` directly into `clean_data[...]`, `NumPy` applies the boolean array as a selective index, replacing only the True positions (`-999.0 and 150.0`) with `25.04` in a single vectorized step.

### Step 3, Statistical Aggregation & Normalization:

Now that our data is clean, let's process spatial and temporal statistics:
- Spatial Aggregation (Per-Sensor Trend): Compute the average reading for each sensor across all time steps.
- Standardization ($Z$-Score Normalization): Scale the clean dataset so that each sensor's time-series has a mean of $0$ and a standard deviation of $1$:$$Z = \frac{X - \mu}{\sigma}$$

### Guiding Question:

To compute $Z$-score normalization for each sensor across time, we need each sensor's mean ($\mu$) and standard deviation ($\sigma$) computed down the rows (`axis=0`).How would you write the vectorized expression to compute `z_scores` for all elements in `clean_data` at once using np.mean and `np.std`?

In [ ]:
z_scores = (clean_data - np.mean(clean_data, axis=0)) / np.std(clean_data, axis=0)

By specifying `axis=0`, `NumPy` calculates the mean and standard deviation down the columns (for each of the 5 sensors across time). Through broadcasting, `NumPy` subtracts the `1D` mean array of shape (`5,`) and divides by the `1D` standard deviation array of shape (`5,`) across the 2D array of shape (`10, 5`) seamlessly!

### Step 4, Summary & Pipeline Verification:

Let's review the complete pipeline we just built:Filtering: Created a boolean mask `(raw_data <= 0) | (raw_data >= 100)` to isolate missing data and noise anomalies.
- In-place Cleaning: Replaced anomalous readings directly with the valid scalar median.Normalization: Applied column-wise $Z$-score standardization (`axis=0`) via broadcasting

In [4]:
import numpy as np

# 1. Generate raw temperature readings
np.random.seed(42)

# 10 time steps × 5 sensors
raw_data = np.random.normal(loc=25.0, scale=3.0, size=(10, 5))

# Inject artificial anomalies
raw_data[2, 1] = -999.0
raw_data[7, 3] = 150.0

print("Raw Data:")
print(raw_data)


# 2. Calculate the median of valid readings
valid_mask = (raw_data > 0) & (raw_data < 100)
valid_median = np.median(raw_data[valid_mask])

print("\nMedian of Valid Readings:")
print(valid_median)


# 3. Clean the data
clean_data = raw_data.copy()

# Identify anomalous readings
anomaly_mask = (clean_data <= 0) | (clean_data >= 100)

# Replace anomalies with the valid median
clean_data[anomaly_mask] = valid_median

print("\nCleaned Data:")
print(clean_data)


# 4. Spatial aggregation: average reading per sensor
sensor_means = np.mean(clean_data, axis=0)

print("\nAverage Reading per Sensor:")
print(sensor_means)


# 5. Z-score normalization per sensor
sensor_means = np.mean(clean_data, axis=0)
sensor_stds = np.std(clean_data, axis=0)

z_scores = (clean_data - sensor_means) / sensor_stds

print("\nZ-Score Normalized Data:")
print(z_scores)


# 6. Verify the pipeline
print("\nVerification:")
print("Original data shape:", raw_data.shape)
print("Cleaned data shape:", clean_data.shape)
print("Normalized data shape:", z_scores.shape)
print("Number of anomalies replaced:", np.sum(anomaly_mask))
print("Normalized column means:", np.mean(z_scores, axis=0))
print("Normalized column standard deviations:", np.std(z_scores, axis=0))

Raw Data:
[[  26.49014246   24.5852071    26.94306561   29.56908957   24.29753988]
 [  24.29758913   29.73763845   27.30230419   23.59157684   26.62768013]
 [  23.60974692 -999.           25.72588681   19.26015927   19.8252465 ]
 [  23.31313741   21.96150664   25.942742     22.27592777   20.7630889 ]
 [  29.39694631   24.3226711    25.20258461   20.72575544   23.36685183]
 [  25.33276777   21.54701927   26.12709406   23.19808393   24.12491875]
 [  23.19488016   30.55683455   24.95950833   21.82686721   27.46763474]
 [  21.33746905   25.62659079   19.12098963  150.           25.59058371]
 [  27.21539974   25.51410484   24.65305515   24.09668891   20.56443403]
 [  22.84046737   23.61808369   28.17136668   26.03085487   19.71087953]]

Median of Valid Readings:
24.310130113846427

Cleaned Data:
[[26.49014246 24.5852071  26.94306561 29.56908957 24.29753988]
 [24.29758913 29.73763845 27.30230419 23.59157684 26.62768013]
 [23.60974692 24.31013011 25.72588681 19.26015927 19.8252465 ]
 [23.3131

What you should observe
- Cleaning: Both `-999.0` and `150.0` are replaced with the median of valid readings.
- Spatial aggregation: sensor_means contains five values, one for each sensor.
- Normalization: `z_scores` retains the shape (`10, 5`).
- Verification: Each normalized sensor column should have a mean approximately equal to `0` and a standard deviation approximately equal to 1.

Key concept: `axis=0` calculates statistics down the rows, producing one result per sensor (`column`). `NumPy` broadcasting then applies those five means and standard deviations to the corresponding columns automatically